# Bài học 19: Giao diện Chat — Không gian làm việc chính

## Từ kiến trúc đến hội thoại

Bài trước, bạn đã theo dõi cách mọi thứ kết nối bên trong. Giờ cùng xem **phía người dùng** — giao diện chat.

**Giao diện chat** là **cách chính để sử dụng sản phẩm này**. Chỉ cần chạy `python output/chat.py` và nói chuyện tự nhiên:

- *"Tạo một bài viết về on-page SEO"*
- *"Bài viết đó xong chưa?"*
- *"Chạy lại bài viết số 3"*
- *"Tải chủ đề từ file my-topics.csv"*
- *"Cho tôi xem tất cả bài viết đang chờ duyệt"*

AI hiểu ý định của bạn và tự động thực hiện — không cần thuộc cú pháp lệnh.

## Agno Team — Một đội AI chuyên biệt

Giao diện chat được xây dựng với **Agno Team** — một nhóm các agent AI, mỗi agent có một vai trò riêng.

Cách hoạt động:
1. Bạn gửi một tin nhắn
2. **Leader** (trưởng nhóm) đọc và hiểu yêu cầu của bạn
3. Leader **giao việc** cho thành viên phù hợp
4. Thành viên thực hiện công việc (gọi tools) và trả kết quả
5. Leader **tổng hợp** và phản hồi lại cho bạn

Giống như một **quản lý dự án** thực thụ — bạn nói muốn gì, PM biết giao cho ai.

In [ ]:
print("""
SEO Workspace Team:
  
  Leader (Claude Opus) — Nhận yêu cầu, giao việc cho thành viên
    |
    +-- Content Creator (Claude Sonnet)
    |     Tools: create_article, create_article_batch,
    |            retry_article, load_articles_from_csv
    |     Vai trò: Tạo bài viết mới, chạy lại bài lỗi, tải chủ đề từ CSV
    |
    +-- Status Tracker (Claude Sonnet)
    |     Tools: list_all_articles, get_article_details, 
    |            get_article_content, get_version_history
    |     Vai trò: Kiểm tra trạng thái bài viết
    |
    +-- SEO Manager (Claude Sonnet)
          Tools: check_rankings, set_article_published_url,
                 get_ranking_history
          Vai trò: URL xuất bản, theo dõi thứ hạng

Ví dụ hội thoại:
  Bạn: "Tạo một bài viết về on-page SEO"
  -> Leader giao cho Content Creator
  -> Content Creator chạy pipeline
  -> Leader: "Đã tạo bài viết recABC123, 2000 từ, file: content/seo-on-page.md"

  Bạn: "Đặt URL là https://blog.example.com/on-page-seo"
  -> Leader giao cho SEO Manager
  -> SEO Manager đặt published_url cho bài viết

  Bạn: "Kiểm tra thứ hạng cho bài viết đó"
  -> Leader giao cho SEO Manager
  -> SEO Manager gọi DataForSEO và trả về vị trí
""")

## Workspace Tools

Các thành viên gọi **workspace tools** để thực hiện công việc. Đây là các hàm Python thuần túy trong `tools/workspace.py`:

| Tool | Thành viên | Mục đích |
|------|--------|--------|
| `create_article` | Content Creator | Tạo 1 bài viết (chạy pipeline) |
| `create_article_batch` | Content Creator | Tạo nhiều bài viết cùng lúc |
| `retry_article` | Content Creator | Chạy lại bài viết bị lỗi |
| `load_articles_from_csv` | Content Creator | Tải chủ đề từ file CSV |
| `list_all_articles` | Status Tracker | Xem tất cả bài viết |
| `get_article_details` | Status Tracker | Xem chi tiết 1 bài viết |
| `get_article_content` | Status Tracker | Xem nội dung bài viết |
| `get_version_history` | Status Tracker | Xem lịch sử phiên bản |
| `check_rankings` | SEO Manager | Kiểm tra vị trí SERP qua DataForSEO |
| `set_article_published_url` | SEO Manager | Đặt URL đã xuất bản |
| `get_ranking_history` | SEO Manager | Xem dữ liệu thứ hạng theo thời gian |

Đây chính là những hàm bạn đã học trong các bài trước — chỉ được đóng gói lại để các thành viên trong team dùng làm tools.

## Ghi chú: json.loads() và json.dumps()

Bạn sẽ thấy hai hàm này khi làm việc với dữ liệu JSON trong Python:

- `json.loads(text)` — **load string**: chuyển chuỗi JSON thành dict/list Python
- `json.dumps(data)` — **dump string**: chuyển dict/list Python thành chuỗi JSON

```python
import json

# Chuỗi JSON -> dict Python
text = '{"title": "SEO Guide", "words": 1500}'
data = json.loads(text)
print(data["title"])  # "SEO Guide"

# Dict Python -> chuỗi JSON
back_to_text = json.dumps(data)
print(back_to_text)   # '{"title": "SEO Guide", "words": 1500}'
```

Chữ 's' trong loads/dumps là viết tắt của 'string'. Ngoài ra còn có `json.load(file)` / `json.dump(data, file)` để đọc/ghi file trực tiếp.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../../output"))

from tools.workspace import (
    create_article,
    create_article_batch,
    retry_article,
    load_articles_from_csv,
    list_all_articles,
    get_article_details,
    get_article_content,
    get_version_history,
    check_rankings,
    set_article_published_url,
    get_ranking_history,
)

# Đây là các hàm Python thuần túy!
# Các thành viên trong team gọi chúng như tools

# Ví dụ: xem tất cả bài viết
import json
result = list_all_articles()
articles = json.loads(result)
print(f"Tìm thấy {len(articles)} bài viết trong workspace")
for a in articles:
    print(f"  {a['id']}: {a['topic']} ({a['status']})")

In [ ]:
# Ví dụ: Tools của SEO Manager (không tốn API)
# Đây là những hàm mà agent SEO Manager gọi

# Xem lịch sử thứ hạng (đọc từ Airtable)
result = get_ranking_history("recXXX")
print("Lịch sử thứ hạng bài viết recXXX:")
print(result)

## Chạy giao diện chat

Để bắt đầu chat, chạy từ thư mục gốc dự án:

```bash
python output/chat.py
```

Khi chạy, bạn sẽ thấy:
- Lời chào và hướng dẫn sử dụng
- Dấu `>` để bạn nhập tin nhắn
- Phản hồi và hành động của AI
- Gõ `quit` hoặc `exit` để thoát

Lịch sử chat được lưu trong `chat_sessions.db`, nên lần sau bạn có thể tiếp tục từ chỗ dừng lại.

In [ ]:
print("Để chạy giao diện chat:")
print()
print("  python output/chat.py")
print()
print("Lịch sử chat được lưu trong chat_sessions.db")
print("nên bạn có thể tiếp tục từ chỗ dừng lại.")

## Đọc code sản phẩm

Bạn đã thấy mọi khái niệm qua các notebook. Code sản phẩm trong `output/` sử dụng **đúng những pattern đó** — đây là nơi tìm từng phần:

### Bản đồ: Bài học → File sản phẩm

| Bạn đã học | Bài học | File sản phẩm |
|---|---|---|
| Cách LLM hoạt động, prompt, model | 05-07 | (Kiến thức nền — định hướng mọi quyết định) |
| Tạo agent (name, model, instructions, tools) | 08-09 | `output/agents/researcher.py`, `outliner.py`, `writer.py`, `image.py` |
| Pydantic schema (ContentOutline, EnrichedContent) | 10, 14 | `output/agents/schemas.py` |
| Nối chuỗi agent, mini pipeline, pipeline đầy đủ | 11-12, 14-16 | `output/pipeline.py` |
| Custom toolkit (FreepikTools, DataForSEOTools) | 09, 15 | `output/agents/image.py` |
| Database (Airtable, các hàm CRUD) | 17 | `output/tools/airtable.py` |
| Cách mọi thứ kết nối | 18 | Tất cả file — toàn bộ luồng yêu cầu |
| Agno Team (leader + member + workspace tools) | 19 | `output/chat.py` + `output/tools/workspace.py` |

### Cấu trúc file

```
output/
├── chat.py                     # Giao diện chat — ĐIỂM VÀO chính (bài 19)
├── pipeline.py                 # Pipeline 4 bước (bài 16)
├── agents/                     # Các AI worker + schema + toolkit (bài 8-15)
│   ├── __init__.py             # Re-exports tất cả agent và schema
│   ├── schemas.py              # Pydantic schema (ContentOutline, EnrichedContent, ...)
│   ├── researcher.py           # Research agent (Claude Sonnet + DuckDuckGo)
│   ├── outliner.py             # Outline agent (Claude Sonnet + output_schema)
│   ├── writer.py               # Writer agent (Grok-4, plain Markdown)
│   ├── image.py                # Image agent + FreepikTools + DataForSEOTools
│   ├── content_creator.py      # Content Creator (thành viên team)
│   ├── status_tracker.py       # Status Tracker (thành viên team)
│   ├── seo_manager.py          # SEO Manager (thành viên team)
│   └── team.py                 # Agno Team (leader + 3 thành viên)
└── tools/                      # API bên ngoài + tích hợp
    ├── __init__.py             # Re-exports
    ├── airtable.py             # Database chính — Airtable (bài 17)
    ├── workspace.py            # Tools cho thành viên team (bài 19)
    └── rankings.py             # Kiểm tra thứ hạng SERP qua DataForSEO
```

**Mẹo:** Khi muốn sửa gì đó, dùng bảng trên để tìm đúng file. Ví dụ, để thay đổi hướng dẫn cho writer, mở `output/agents/writer.py` và sửa danh sách instructions của `writer_agent`.

## Bài tập

Điền vào chỗ trống để gọi workspace tools và khám phá workspace:

1. Dùng `get_article_details(article_id)` để lấy chi tiết một bài viết (chọn một ID từ danh sách phía trên)
2. Parse kết quả JSON và in ra topic, status và số từ
3. Nếu bài viết có nội dung, dùng `get_article_content(article_id)` để đọc 500 ký tự đầu tiên

Đây chính là những gì Status Tracker agent làm khi bạn hỏi "Trạng thái bài viết số 3 thế nào?" — bạn đang gọi đúng những hàm mà nó gọi.

In [ ]:
# Bài tập: Điền vào chỗ trống để khám phá workspace

import json

# 1. Lấy chi tiết bài viết (chọn Airtable record ID từ danh sách phía trên)
article_id = "recXXX"  # Thay bằng Airtable record ID thực tế
result = ___(article_id)

# 2. Parse kết quả JSON thành dict Python
article = json.___(result)

# 3. In ra topic, status và số từ
print(f"Chủ đề: {article[___]}")
print(f"Trạng thái: {article[___]}")
print(f"Số từ: {article.get(___, 'N/A')}")

# 4. Nếu có nội dung, đọc 500 ký tự đầu tiên
content = ___(article_id)
if content and not content.startswith("Article"):
    print(f"\n500 ký tự đầu:\n{content[:___]}")

<details>
<summary>Bấm để xem đáp án</summary>

```python
import json

# 1. Lấy chi tiết bài viết (chọn Airtable record ID từ danh sách phía trên)
article_id = "recXXX"  # Thay bằng Airtable record ID thực tế
result = get_article_details(article_id)

# 2. Parse kết quả JSON thành dict Python
article = json.loads(result)

# 3. In ra topic, status và số từ
print(f"Chủ đề: {article['topic']}")
print(f"Trạng thái: {article['status']}")
print(f"Số từ: {article.get('word_count', 'N/A')}")

# 4. Nếu có nội dung, đọc 500 ký tự đầu tiên
content = get_article_content(article_id)
if content and not content.startswith("Article"):
    print(f"\n500 ký tự đầu:\n{content[:500]}")
```

**Lưu ý:** `get_article_details()` trả về JSON (nên dùng `json.loads()`), nhưng `get_article_content()` trả về văn bản Markdown trực tiếp — không cần parse JSON.
</details>

## Tổng kết

Bạn đã hiểu **toàn bộ sản phẩm** — từ lưu trữ database đến kiến trúc đến đội AI hội thoại.

Đây là những gì bạn đã học:

### Mô-đun 1: Nền tảng Python (Bài 1-4)
Biến, list, dictionary, hàm, package — nền móng cơ bản.

### Mô-đun 2: Hiểu về AI (Bài 5-7)
Cách LLM hoạt động, prompt và ngữ cảnh, lựa chọn model — cái "tại sao" đằng sau mọi thứ.

### Mô-đun 3: Xây dựng Agent (Bài 8-12)
Agent đầu tiên, tools, structured output, nối chuỗi, mini pipeline — thực hành xây dựng.

### Mô-đun 4: Xây dựng sản phẩm (Bài 13-16)
Claude Code, agent nghiên cứu, dàn ý, viết bài, hình ảnh — từ công cụ đến pipeline.

### Mô-đun 5: Sản phẩm hoàn chỉnh (Bài 17-20)
Database (Airtable), cách mọi thứ kết nối, giao diện chat, mở rộng sản phẩm.

**Bài tiếp theo:** Thực hành ví dụ thực tế — thêm agent mới vào pipeline bằng Claude Code.